In [2]:
from pathlib import Path
import sys

# Go from notebooks/ to project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

c:\MedVision-AI


# MedVision-AI

# Notebook 07

# EfficientNet-B0 Model Implementation

---

## Objectives

In this notebook, we prepare the deep learning model that will be used for pneumonia classification.

Unlike the previous notebook, which focused on the theoretical concepts of CNNs and Transfer Learning, this notebook focuses on implementing a production-ready EfficientNet-B0 architecture using PyTorch.

### Learning Goals

- Import the required deep learning libraries.
- Configure project constants.
- Load the pretrained EfficientNet-B0 model.
- Explore the model architecture.
- Understand the original ImageNet classifier.
- Replace the classifier for binary classification.
- Freeze pretrained feature extraction layers.
- Move the model to the appropriate device.
- Verify trainable and frozen parameters.
- Perform a dummy forward pass.
- Save the initialized model.

> **Note:** No model training is performed in this notebook. The goal is to prepare and verify the model before building the training pipeline.

In [5]:
import torch

from app.models.efficientnet import build_efficientnet_b0

# Configuration

Centralizing project constants makes the notebook easier to maintain and ensures that future notebooks use consistent settings.

In [6]:
# =====================================
# Configuration
# =====================================

NUM_CLASSES = 2
IMAGE_SIZE = 224

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using Device : {DEVICE}")

Using Device : cuda


# Section 1

# Load Pretrained EfficientNet-B0

##  Objective

Build the EfficientNet-B0 model using the reusable model builder from app/models/efficientnet.py. The builder loads pretrained ImageNet weights and configures the model for binary classification.

Using pretrained weights allows us to reuse visual features learned from millions of images instead of training a CNN from scratch.

In [8]:
model = build_efficientnet_b0(
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_backbone=True,
)

In [9]:
print(type(model).__name__)

EfficientNet


# Section 2

# Explore Model Architecture

##  Objective

Understand the high-level structure of EfficientNet-B0.

The model consists of three major components:

- Feature Extractor
- Average Pooling
- Classifier

For transfer learning, we keep the feature extractor and replace only the classifier.

In [5]:
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [6]:
print(model.features)

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
  )
  (1): Sequential(
    (0): MBConv(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (activation): SiLU(inplace=True)
          (scale_activation): Sigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), 

The classifier replacement is implemented inside the reusable model builder. Here we inspect the customized classifier that has been attached to the model

In [7]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


# Section 3

# Understanding the Original Classifier

EfficientNet-B0 was originally trained on the ImageNet dataset.

The original classifier predicts one of **1000 classes**.

Our dataset contains only two classes:

- NORMAL
- PNEUMONIA

Therefore, we must replace the original classification layer.

In [8]:
num_features = model.classifier[1].in_features

print(f"Input Features : {num_features}")

Input Features : 1280


In [10]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


# Section 4 

# Move Model to Device

Move the model to the available computation device (GPU or CPU).

This ensures that future training and inference are performed on the same device.

In [12]:
model = model.to(DEVICE)

print(next(model.parameters()).device)

cuda:0


# Section 5

# Count Parameters

## Objective

Verify how many parameters are trainable after freezing the feature extractor.

In [13]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {frozen_params:,}")

Total Parameters      : 4,010,110
Trainable Parameters  : 2,562
Frozen Parameters     : 4,007,548


# Section 6

# Dummy Forward Pass

## Objective

Verify that the modified model accepts an input tensor and produces the expected output shape.

This step confirms that the model architecture has been configured correctly before training.

In [14]:
model.eval()

with torch.no_grad():

    dummy_input = torch.randn(
        4,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=DEVICE,
    )

    output = model(dummy_input)

print(output.shape)

torch.Size([4, 2])


## Interpretation

The model successfully processed a batch of four images.

For each image, the model produced two output scores corresponding to:

- NORMAL
- PNEUMONIA

This verifies that the classifier replacement was successful.

# Section 7

# Model Summary

We have successfully:

- Loaded EfficientNet-B0.
- Loaded pretrained ImageNet weights.
- Explored the architecture.
- Replaced the classifier.
- Frozen pretrained layers.
- Moved the model to the appropriate device.
- Verified trainable parameters.
- Verified the forward pass.

The model is now fully prepared for training.

# Section 8

# Save Initialized Model 

Saving the initialized model provides a reproducible starting point for future experiments.

This model has **not** been trained yet. It only contains the pretrained EfficientNet backbone and the newly initialized classifier.

In [15]:
from pathlib import Path

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

checkpoint_path = CHECKPOINT_DIR / "efficientnet_b0_initialized.pth"

torch.save(
    model.state_dict(),
    checkpoint_path,
)

print(f"Model saved to: {checkpoint_path}")

Model saved to: checkpoints\efficientnet_b0_initialized.pth


# Notebook Conclusion

In this notebook, we transformed a pretrained EfficientNet-B0 model into a binary image classifier suitable for pneumonia detection.

### Key Takeaways

- Loaded the pretrained EfficientNet-B0 architecture.
- Explored its feature extractor and classifier.
- Replaced the original ImageNet classifier with a custom binary classifier.
- Froze the pretrained backbone while keeping the classifier trainable.
- Verified the model using a dummy forward pass.
- Saved the initialized model for reproducibility.

The model is now fully prepared for the next stage of the project.

---

